# 排序穩定性實驗室
Status: in_use  
Prereq: Python 基礎語法、input()/split()、for 迴圈  
Can-do checklist:  
- [ ] 能用例子解釋「排序穩定性」
- [ ] 能設計測資驗證排序是否穩定
- [ ] 能在多鍵排序時正確處理優先順序


在排序演算法中，若多個元素的比較鍵相同，**穩定排序**會保留它們原本的相對順序；**不穩定排序**則可能打亂這些元素的位置。這份 Notebook 讓你動手觀察兩者差異，並練習如何把不穩定演算法改寫成穩定版本。


## 目標
- 重溫穩定排序的定義與應用情境。
- 實際執行穩定與不穩定排序，觀察相同鍵值的元素是否交換。
- 嘗試為不穩定的堆排序加入索引，使其輸出穩定。


## 建立測試資料
我們用一份「學生、成績、座號」的列表，每位同學的成績可能相同，但座號固定。請執行下方程式建立資料與工具函式。


In [1]:
from dataclasses import dataclass
from typing import List

@dataclass
class Student:
    name: str
    score: int
    seat: int

# 刻意打亂分數與座號順序，方便觀察穩定與不穩定排序的差異
sample_students: List[Student] = [
    Student("Blair", 92, 1),
    Student("Casey", 86, 2),
    Student("Drew", 92, 5),
    Student("Alex", 86, 4),
    Student("Evan", 86, 8),
]

def describe(students: List[Student]) -> None:
    for stu in students:
        print(f"{stu.name: <5} score={stu.score} seat={stu.seat}")


### 觀察內建穩定排序
Python 的 `sorted` 採 Timsort。下列範例僅以成績作為排序鍵，但由於演算法穩定，相同分數的學生會保留原本出現順序（等同於以原座號作為第二排序依據）。


In [2]:
sorted_students = sorted(sample_students, key=lambda s: s.score)
print("原始資料：")
describe(sample_students)
print("穩定排序結果：")
describe(sorted_students)


原始資料：
Blair score=92 seat=1
Casey score=86 seat=2
Drew  score=92 seat=5
Alex  score=86 seat=4
Evan  score=86 seat=8
穩定排序結果：
Casey score=86 seat=2
Alex  score=86 seat=4
Evan  score=86 seat=8
Blair score=92 seat=1
Drew  score=92 seat=5


### 實驗：不穩定堆排序
以下函式透過 `heapq` 建立最小堆，刻意以姓名當作次要排序鍵，觀察同分同學的相對順序如何被打亂。


In [3]:
import heapq
from typing import Iterable

def heap_sort_unstable(students: Iterable[Student]) -> List[Student]:
    # use name as secondary key to illustrate instability
    heap = [(stu.score, stu.name, stu) for stu in students]
    heapq.heapify(heap)
    ordered = []
    while heap:
        _, _, stu = heapq.heappop(heap)
        ordered.append(stu)
    return ordered

print("原始資料：")
describe(sample_students)
print("不穩定堆排序結果（同分改用姓名）：")
describe(heap_sort_unstable(sample_students))


原始資料：
Blair score=92 seat=1
Casey score=86 seat=2
Drew  score=92 seat=5
Alex  score=86 seat=4
Evan  score=86 seat=8
不穩定堆排序結果（同分改用姓名）：
Alex  score=86 seat=4
Casey score=86 seat=2
Evan  score=86 seat=8
Blair score=92 seat=1
Drew  score=92 seat=5


### 改寫：加入原始索引維持穩定
在堆的鍵值中加入原始位置（或座號），可以保留同分同學的出現順序，讓堆排序變成穩定版本。


In [4]:
def heap_sort_stable(students: Iterable[Student]) -> List[Student]:
    heap = [(stu.score, idx, stu) for idx, stu in enumerate(students)]
    heapq.heapify(heap)
    ordered = []
    while heap:
        _, _, stu = heapq.heappop(heap)
        ordered.append(stu)
    return ordered

print("原始資料：")
describe(sample_students)
print("加入原始索引後的穩定堆排序：")
describe(heap_sort_stable(sample_students))


原始資料：
Blair score=92 seat=1
Casey score=86 seat=2
Drew  score=92 seat=5
Alex  score=86 seat=4
Evan  score=86 seat=8
加入原始索引後的穩定堆排序：
Casey score=86 seat=2
Alex  score=86 seat=4
Evan  score=86 seat=8
Blair score=92 seat=1
Drew  score=92 seat=5


## 挑戰任務
1. 修改 `heap_sort_unstable`，加入「次要鍵值」例如座號，讓排序結果恢復穩定。
2. 自行撰寫選擇排序（Selection Sort），觀察它是否穩定。若不穩定，思考可否在交換時加上條件維持穩定性。
3. 設計另一份測試資料（例如：商品、價格、上架順序），重複以上實驗。

完成後可將你的成果貼回主筆記，或整理成課堂分享。
